Here I test the model CLIP given different images.

The objective is to find the bounding box of the relative 
label given the image.

With CLIP we can compute the score 

In [2]:
import numpy as np
import torch
import clip

In [3]:
print('available models: ',clip.available_models())

available models:  ['RN50', 'RN101', 'RN50x4', 'RN50x16', 'RN50x64', 'ViT-B/32', 'ViT-B/16', 'ViT-L/14', 'ViT-L/14@336px']


In [6]:
model, preprocess = clip.load("RN50")

In [7]:
if torch.backends.mps.is_available():
    print("MPS backend is available.")
    device = torch.device('mps')
elif torch.cuda.is_available():
    print("CUDA backend is available.")
    device = torch.device('cuda')
else:
    print("Neither CUDA or MPS backend are available. Resorting to CPU")
    device = torch.device('cpu')

MPS backend is available.


In [8]:
model.to(device).eval()

CLIP(
  (visual): ModifiedResNet(
    (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu1): ReLU(inplace=True)
    (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu2): ReLU(inplace=True)
    (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu3): ReLU(inplace=True)
    (avgpool): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
     

In [9]:
input_resolution = model.visual.input_resolution

context_length = model.context_length

vocab_size = model.vocab_size

In [12]:
print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Input resolution:", input_resolution)
print("Context length:", context_length)
print("Vocab size:", vocab_size)

print('Tokenization of the sentence: "vamos a bailar" ')
print(clip.tokenize('vamos a bailar'))



Model parameters: 102,007,137
Input resolution: 224
Context length: 77
Vocab size: 49408
Tokenization of the sentence: "vamos a bailar" 
tensor([[49406, 30346,   320, 22933,   625, 49407,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0]], dtype=torch.int32)


In [14]:
from ultralytics import YOLO

In [15]:
yolo_model = YOLO("yolov8n.pt")
yolo_model.to(device)


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_s

In [26]:

# Load image
imgs = ['https://ultralytics.com/images/zidane.jpg']  # batch of images

# Preprocess image
results = yolo_model(imgs)



0: 384x640 2 persons, 1 tie, 72.2ms
Speed: 3.6ms preprocess, 72.2ms inference, 8.6ms postprocess per image at shape (1, 3, 384, 640)


In [27]:
results

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted p

In [1]:
boxes = results[0].boxes.data

NameError: name 'results' is not defined

In [32]:
boxes

tensor([[1.1487e+02, 1.9741e+02, 1.1145e+03, 7.1189e+02, 8.3597e-01, 0.0000e+00],
        [7.4846e+02, 4.1855e+01, 1.1431e+03, 7.1302e+02, 8.1896e-01, 0.0000e+00],
        [4.3947e+02, 4.3707e+02, 5.2435e+02, 7.0916e+02, 2.9097e-01, 2.7000e+01]], device='mps:0')

In [34]:
help(results[0])

Help on Results in module ultralytics.engine.results object:

class Results(ultralytics.utils.SimpleClass)
 |  Results(orig_img, path, names, boxes=None, masks=None, probs=None, keypoints=None, obb=None, speed=None) -> None
 |  
 |  A class for storing and manipulating inference results.
 |  
 |  Attributes:
 |      orig_img (numpy.ndarray): Original image as a numpy array.
 |      orig_shape (tuple): Original image shape in (height, width) format.
 |      boxes (Boxes, optional): Object containing detection bounding boxes.
 |      masks (Masks, optional): Object containing detection masks.
 |      probs (Probs, optional): Object containing class probabilities for classification tasks.
 |      keypoints (Keypoints, optional): Object containing detected keypoints for each object.
 |      speed (dict): Dictionary of preprocess, inference, and postprocess speeds (ms/image).
 |      names (dict): Dictionary of class names.
 |      path (str): Path to the image file.
 |  
 |  Methods:
 |   

In [38]:
# Results
results[0].show()  # or .show()

curl: (23) Failure writing output to destination
curl: (23) Failure writing output to destination
curl: (23) Failure writing output to destination
